# `pat2vec`: Example Searches Test

This notebook demonstrates how to use the individual search components within `pat2vec`. Each cell tests a specific `search_*` function, which is responsible for retrieving a particular type of data (e.g., demographics, bloods, appointments) from the data source.

This is useful for:
- Verifying that your connection to Elasticsearch is working correctly (when `testing=False`).
- Understanding the structure of the data returned by each component.
- Debugging individual data retrieval steps.

## 1. Define a Sample Cohort

First, we load a sample CSV to get a small list of patient IDs. In a real-world scenario, this would be your study cohort. We'll use the first 5 patients for this example.

In [ ]:
import pandas as pd

df = pd.read_csv("test_files/treatment_docs.csv")

all_patient_list = df["client_idcode"].to_list()[0:5]

all_patient_list

## 2. Initialize `pat2vec`

Next, we clean up any previous test runs and initialize the main `pat2vec` object.

- **`testing=True`**: This is a crucial parameter. When `True`, `pat2vec` uses a built-in dummy data generator instead of connecting to a live Elasticsearch instance. This allows you to run the notebook and test the logic without needing credentials or a database connection. Set this to `False` to test against your actual data.
- **`cogstack=True`**: This configures the object to use Cohort Searcher methods designed for a CogStack-like environment.

In [ ]:
!rm -r example_searches_test

import pat2vec

from pat2vec.util.config_pat2vec import config_class
from pat2vec.main_pat2vec import main


config_obj = config_class(
    testing=True,
    verbosity=9,
    proj_name="example_searches_test",
    # credentials_path = '../gloabl_files/credentials.py'
)  # Use defaults, set testing unless in live environment with cogstack instance.

pat2vec_obj = main(
    config_obj=config_obj, cogstack=True
)  # initialize the pat2vec object with defaults

## 3. Exploring Helper Functions

`pat2vec` includes several helper functions to make it easier to understand how it interacts with your data indices.

### Inspecting Method-to-Index Mappings

Each data retrieval function in `pat2vec` (e.g., `get_current_pat_bloods`) is mapped to a specific Elasticsearch index. You can use the following helpers to see these mappings.

In [ ]:

# Get the index for a single method
bloods_index = pat2vec.get_index_for_method('get_current_pat_bloods')
print(f"The 'get_current_pat_bloods' function queries the '{bloods_index}' index by default.")

# Get a complete mapping of all methods to their indices
all_indices = pat2vec.get_all_method_indices()

print("\nDefault indices for all get methods:")
for method, index in all_indices.items():
    print(f"- {method}: {index}")

# You can also access the dictionary directly
print(pat2vec.GET_METHOD_INDEX_MAP)


### Inspecting Data Fields

You can also inspect which data fields are returned by default for each method, or query the index to see all available fields.

**Note**: To get *all* available fields (`get_all_fields_for_method`), the notebook must be able to connect to an Elasticsearch client. When `testing=True`, this works with the dummy client. When `testing=False`, you must have your `credentials.py` file configured correctly.

In [ ]:
# Get the default fields for the 'get_current_pat_bloods' method
bloods_default_fields = pat2vec.get_default_fields_for_method('get_current_pat_bloods')
print(f"Default fields for 'get_current_pat_bloods':\n{bloods_default_fields}\n")

# You can also get the entire map of methods to their default fields
all_default_fields = pat2vec.get_all_method_default_fields()
# print(all_default_fields)

# To get all available fields, you first need to initialize the Elasticsearch client
# (make sure your credentials.py is configured)
pat2vec.initialize_cogstack_client()

# Now, get all available fields for the index used by 'get_current_pat_bloods'
bloods_all_fields = pat2vec.get_all_fields_for_method('get_current_pat_bloods')
print(f"All available fields for 'get_current_pat_bloods' (from index 'basic_observations'):\n{bloods_all_fields}\n")


## 4. Testing Individual Search Methods

The following cells demonstrate how to call each specific search function. Each function takes the `cohort_searcher_with_terms_and_search` object (which handles the Elasticsearch query logic) and our list of patient IDs.

The output of each cell is a pandas DataFrame containing the retrieved data for our sample cohort.

Understanding the search_appointments Function
The search_appointments function is a specialized method designed for retrieving appointment data from Elasticsearch. It acts as a convenient wrapper around the more generic cohort_searcher_with_terms_and_search function, pre-configuring several options to make searching for appointments straightforward. Despite its convenience, it provides a rich set of parameters to customize the search to your specific needs.

How It Works
The function constructs an Elasticsearch query based on the arguments you provide. The core of this query consists of:

A filter for a specific list of patient IDs (client_id_codes).
A date range filter based on the appointments_time_field.
You can further refine this base search by using the additional_custom_search_string parameter to inject custom query logic.

Parameters Explained
Below is a breakdown of the arguments for search_appointments, grouped by their function. You can modify these default values to suit your specific research question.

1. Required Parameters

cohort_searcher_with_terms_and_search: This is the underlying search function from pat2vec that handles communication with Elasticsearch. You must pass the initialized searcher object here (e.g., pat2vec_obj.cohort_searcher_with_terms_and_search).
client_id_codes: A list of patient identifiers you want to find appointments for. Internally, the function filters on the HospitalID.keyword field to match these IDs.
2. Time-based Filtering

appointments_time_field: The specific date field in the Elasticsearch index to use for filtering. Defaults to "AppointmentDateTime".
start_year, start_month, start_day: Define the beginning of the search window. Defaults to 1995-01-01.
end_year, end_month, end_day: Define the end of the search window. Defaults to 2025-12-12.
3. Data & Index Customization

fields_override: By default, the function returns a pre-defined list of APPOINTMENT_FIELDS (e.g., ClinicCode, Attended, Specialty). You can provide your own list of field names here to retrieve only the specific data you need. Defaults to None.
index_name: The Elasticsearch index pattern to search against. Defaults to "pims_apps*".
additional_custom_search_string: This allows you to add any valid Elasticsearch query string syntax to further filter the results.
Example: Use 'AND ClinicDesc:"Cardiology"' to only retrieve appointments within the Cardiology clinic.
Defaults to None.

### Appointments

In [ ]:
# Import the search function
from pat2vec.pat2vec_get_methods.get_method_appointments import search_appointments

# Call the function with all default arguments explicitly shown
df_appointments = search_appointments(
    cohort_searcher_with_terms_and_search=pat2vec_obj.cohort_searcher_with_terms_and_search,
    client_id_codes=all_patient_list,
    appointments_time_field="AppointmentDateTime",
    fields_override=None,  # Or, for example: ['PatNHSNo', 'AppointmentType', 'ClinicDesc']
    start_year="1995",
    start_month="01",
    start_day="01",
    end_year="2025",
    end_month="12",
    end_day="31",
    additional_custom_search_string='AND Attended:"1"', # Example: only attended appointments
    index_name="pims_apps*",
)

df_appointments.head()


### Bed / Location Data

In [ ]:
df_bed = pat2vec.pat2vec_get_methods.get_method_bed.search_bed_data(
    cohort_searcher_with_terms_and_search=pat2vec_obj.cohort_searcher_with_terms_and_search,
    client_id_codes=all_patient_list,
)

df_bed.head()

### Bloods / Observations

In [ ]:
df_bloods = pat2vec.pat2vec_get_methods.get_method_bloods.search_bloods_data(
    cohort_searcher_with_terms_and_search=pat2vec_obj.cohort_searcher_with_terms_and_search,
    client_id_codes=all_patient_list,
)

df_bloods.head()

### BMI

In [ ]:
df_bmi = pat2vec.pat2vec_get_methods.get_method_bmi.search_bmi_observations(
    cohort_searcher_with_terms_and_search=pat2vec_obj.cohort_searcher_with_terms_and_search,
    client_id_codes=all_patient_list,
)

df_bmi.head()

### Core O2

In [ ]:
df_core_o2 = pat2vec.pat2vec_get_methods.get_method_core02.search_core_o2_observations(
    cohort_searcher_with_terms_and_search=pat2vec_obj.cohort_searcher_with_terms_and_search,
    client_id_codes=all_patient_list,
)

df_core_o2.head()

### Core Resus

In [ ]:
df_core_resus = pat2vec.pat2vec_get_methods.get_method_core_resus.search_core_resus_observations(
    cohort_searcher_with_terms_and_search=pat2vec_obj.cohort_searcher_with_terms_and_search,
    client_id_codes=all_patient_list,
)

df_core_resus.head()

### Demographics

In [ ]:
df_demo = pat2vec.pat2vec_get_methods.get_method_demo.search_demographics(
    cohort_searcher_with_terms_and_search=pat2vec_obj.cohort_searcher_with_terms_and_search,
    client_id_codes=all_patient_list,
)

df_demo.head()

### Diagnostics (Orders)

In [ ]:
df_diagnostics = pat2vec.pat2vec_get_methods.get_method_diagnostics.search_diagnostic_orders(
    cohort_searcher_with_terms_and_search=pat2vec_obj.cohort_searcher_with_terms_and_search,
    client_id_codes=all_patient_list,
)

df_diagnostics.head()

### Drugs

In [ ]:
df_drugs = pat2vec.pat2vec_get_methods.get_method_drugs.search_drug_orders(
    cohort_searcher_with_terms_and_search=pat2vec_obj.cohort_searcher_with_terms_and_search,
    client_id_codes=all_patient_list,
)

df_drugs.head()

### Hospital Site

In [ ]:
df_hosp_site = pat2vec.pat2vec_get_methods.get_method_hosp_site.search_hospital_site(
    cohort_searcher_with_terms_and_search=pat2vec_obj.cohort_searcher_with_terms_and_search,
    client_id_codes=all_patient_list,
)

df_hosp_site.head()

### Smoking Status

In [ ]:
df_smoking = pat2vec.pat2vec_get_methods.get_method_smoking.search_smoking(
    cohort_searcher_with_terms_and_search=pat2vec_obj.cohort_searcher_with_terms_and_search,
    client_id_codes=all_patient_list,
)

df_smoking.head()

### VTE Status

In [ ]:
df_vte = pat2vec.pat2vec_get_methods.get_method_vte_status.search_vte(
    cohort_searcher_with_terms_and_search=pat2vec_obj.cohort_searcher_with_terms_and_search,
    client_id_codes=all_patient_list,
)

df_vte.head()

COVID

In [ ]:
from pat2vec.pat2vec_get_methods.get_method_covid import search_covid

df_covid = search_covid(
    cohort_searcher_with_terms_and_search=pat2vec_obj.cohort_searcher_with_terms_and_search,
    client_id_codes=all_patient_list,
)

df_covid.head()
